# GPU Accelerated PPE Detection Setup 
**Strict Environment Requirements**:
- Python 3.12
- NVIDIA T4 GPU
- Roboflow `vrishank-umrani-s-workspace` → `ppe-gzzdx-nf8ps` (v1)
- YOLOv8n, 25 epochs, imgsz=640


In [ ]:
!pip install -q ultralytics roboflow

In [ ]:
import sys
import torch

print("Verifying Environment...")

# 1. Enforce Python 3.12
if sys.version_info.major != 3 or sys.version_info.minor != 12:
    raise RuntimeError(f"Strict requirement: Python 3.12 is required. Current is {sys.version_info.major}.{sys.version_info.minor}.")

# 2. Check CUDA / Device availability
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available! Please ensure this is running on the remote GPU (NVIDIA T4).")

gpu_name = torch.cuda.get_device_name(0)
print(f"Verified GPU: {gpu_name}")
if "T4" not in gpu_name:
    print("Warning: Target GPU is not an NVIDIA T4, but proceeding anyway.")

In [ ]:
import os
from roboflow import Roboflow

print("Downloading data from Roboflow...")
rf = Roboflow(api_key="sfGRWRbhP29Y2TV8i6PO") 
project = rf.workspace("vrishank-umrani-s-workspace").project("ppe-gzzdx-nf8ps")
dataset = project.version(1).download("yolov8")

data_path = dataset.location

In [ ]:
from ultralytics import YOLO

print("Starting YOLOv8 Training...")
data_yaml = os.path.join(data_path, "data.yaml")

# Load YOLOv8-Nano
model = YOLO("yolov8n.pt")

# Run training
results = model.train(
    data=data_yaml,
    epochs=25,
    imgsz=640,
    device=0,
    project="runs",
    name="ppe_t4_run",
    exist_ok=True,
    verbose=True
)

run_dir = os.path.join("runs", "ppe_t4_run")

In [ ]:
import shutil
import glob

print("Preparing export artifacts...")
sub_dir = "Final_Submission"
os.makedirs(sub_dir, exist_ok=True)

# Gather best.pt
best_weights = os.path.join(run_dir, "weights", "best.pt")
if os.path.exists(best_weights):
    shutil.copy(best_weights, os.path.join(sub_dir, "best.pt"))
    print(f"Copied best.pt into {sub_dir}")
else:
    print("Warning: best.pt not found.")
    
# Gather all graphical outputs from YOLO (includes results.png, confusion_matrix.png)
images = glob.glob(os.path.join(run_dir, "*.png")) + glob.glob(os.path.join(run_dir, "*.jpg"))
for img_path in images:
    img_name = os.path.basename(img_path)
    shutil.copy(img_path, os.path.join(sub_dir, img_name))
    print(f"Copied {img_name} into {sub_dir}")
    
# Zip the Final_Submission folder
zip_name = f"{sub_dir}.zip"
shutil.make_archive(sub_dir, 'zip', sub_dir)
print(f"Successfully packaged everything into {zip_name}!")

# If on Colab, trigger the automatic download
try:
    from google.colab import files
    files.download(zip_name)
    print("Triggered automated download in browser.")
except ImportError:
    pass